# MS Capital: My Hold-out Was a Lucky Stretch

A historical investigation of a favorable holdout period and a lower external leaderboard score. The numerical results below are retained for traceability. The 21 October validation correction qualifies the original interpretation of the ensemble walk-forward score.

The LightGBM holdout measured +0.15171; the separately submitted v4 ensemble measured +0.12900 on the captured leaderboard. Neither value is a current standing or a trading-profitability claim.

## Validation correction — 21 October 2026

The historical ensemble cosine **+0.14088** was obtained by fitting blend weights and scoring them on the **same fold targets**. It is an optimistic fitted-blend diagnostic, not an unbiased out-of-sample estimate. Agreement with the heuristic period adjustment (+0.14084) does not validate either estimate independently.

The repository now fits blend weights only on **earlier folds**, freezes them before scoring the current fold, and excludes the first fold from ensemble scoring. Historical base-model early stopping used the scored validation fold, so those scores are not fully nested outer-test estimates. Existing results below remain historical and are not silently replaced.

The LightGBM holdout (+0.15171) and submitted v4 ensemble leaderboard result (+0.12900) concern different models and evaluation scopes. This notebook re-executes the published result analysis; it does **not** retrain the full competition pipeline or submit predictions.

**NEXUS applied data science · Senanur Çetin · Non-commercial research**

[Validation protocol](https://github.com/senanurcetin/ms-capital-market-forecasting/blob/main/docs/VALIDATION.md) · [Model card](https://github.com/senanurcetin/ms-capital-market-forecasting/blob/main/MODEL_CARD.md)


In [ ]:
import glob, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

R = Path(sorted(glob.glob("/kaggle/input/**/period_difficulty.csv", recursive=True))[0]).parent
difficulty = pd.read_csv(R / "period_difficulty.csv")
dmeta = json.loads((R / "period_difficulty_meta.json").read_text())
walk = pd.read_csv(R / "walkforward_summary.csv")
adv = pd.read_csv(R / "adversarial_auc.csv")
recency = pd.read_csv(R / "recency.csv")
lb = json.loads((R / "leaderboard.json").read_text())
print(f"hold-out months {dmeta['holdout']}, periods scored: {len(difficulty)}")

## 1. Hold the model fixed and let the calendar vary

A model trained on months 0–34 is scored on every later 3-month block, with three seeds per block.
Only the period changes, so the spread in scores is the spread in *difficulty*.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
ax.bar(difficulty.block, difficulty.cosine, yerr=difficulty.seed_std, color="#8aa4c8")
hold = difficulty.block.str.startswith(str(dmeta["holdout"][0]))
ax.bar(difficulty.block[hold], difficulty.cosine[hold], color="#c4553a", label="hold-out months")
ax.axhline(dmeta["typical_cosine"], color="k", lw=0.8, ls="--", label="typical period")
ax.set_ylabel("cosine (fixed model)"); ax.set_title("same model, different months")
ax.tick_params(axis="x", rotation=60); ax.legend(); plt.tight_layout(); plt.show()
print(f"hold-out block percentile: {dmeta['holdout_percentile']:.0%} | "
      f"hold-out/typical ratio {dmeta['ratio']:.3f}")
print(f"heuristically adjusted hold-out: {dmeta['reported']:.4f} / {dmeta['ratio']:.3f} = {dmeta['debiased']:.4f}")

The holdout period sits at the **83rd percentile** of the measured block score distribution: a favorable stretch for the fixed historical model. The adjustment to 0.1408 is a descriptive heuristic, not an unbiased estimator or confidence bound.

## 2. Historical walk-forward results and their limitations

Base models were evaluated over five chronological periods. The historical ensemble fitted and scored blend weights on the same folds, so its aggregate must be treated as a fitted-blend diagnostic. Early stopping also used each base model's validation fold.

In [ ]:
print(walk.round(4).to_string(index=False))
print("Historical ensemble = same-fold fitted-blend diagnostic; not an unbiased score.")
print(f"Period adjustment heuristic: {dmeta['debiased']:.4f}; not independent validation.")

Numerical agreement between the historical fitted blend and the period-adjustment heuristic does **not** establish independence or validate generalization. A fresh evaluation must keep model selection and blend calibration separate from the scored period.

## 3. Historical distribution diagnostics

Adversarial validation measures distribution separability. Its AUC does not by itself causally explain the performance gap or quantify future prediction quality.

In [ ]:
within = adv[adv.kind == "within-train"]
test = adv[adv.kind == "train-vs-test"]
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(within.distance_months, within.auc, marker="o", label="train block vs block, by distance")
ax.axhline(float(test.auc.iloc[0]), color="#c4553a", ls="--", label=f"test vs reference: {float(test.auc.iloc[0]):.3f}")
ax.set_xlabel("months apart"); ax.set_ylabel("adversarial AUC"); ax.legend(); ax.set_ylim(0.7, 0.95)
plt.tight_layout(); plt.show()

In [ ]:
print(recency.round(4).to_string(index=False))
g2 = recency[recency.gap_to_eval == 2].set_index("arm").cosine
print(f"\nAt the same 2-month gap: all history {g2['cumulative']:.4f} vs a sliding window of the same length "
      f"{g2['fixed_window']:.4f}. Recent months help by being added, not by replacing old ones; "
      "a sliding window never beat the full history here.")

## 4. Where that left me on the leaderboard

In [ ]:
print(f"{lb['n_teams']} teams | my score {lb['our_score']} (rank {lb['our_rank']}) | "
      f"median {lb['median']} | top {lb['best']}  (captured {lb['captured']})")

## Takeaways

- A temporally separated holdout can still be a favorable period; report period variability.
- Fit ensemble weights before the scored period. Same-fold fitting and scoring is optimistic.
- Keep holdout, external leaderboard, base-model CV and fitted-blend diagnostics separate.
- Distribution separability is a diagnostic; it does not independently explain a score gap.
- This rerun reads historical result tables; it does not reproduce training or create a new submission.

Full code and limitations: [MSCapital repository](https://github.com/senanurcetin/ms-capital-market-forecasting).

NEXUS applied data science · Senanur Çetin.